# Faz 0 — Hakem A/B tezgahı (gürültü tabanı + v1/v3 karşılaştırması)

**Amaç:** `data/eval/faz0_ornek_125.csv` (karar tipine göre kotalı 125 sorgu)
üzerinde `gemma4:e4b` ile üç ölçümü tek koşuda almak:

| karşılaştırma | ne ölçer |
|---|---|
| `v1#0` ↔ `v1#1` | **gürültü tabanı** — aynı prompt, aynı havuz, aynı oturum |
| `v1#0` ↔ `v3#0` | **ölü kural temizliğinin etkisi** (prompt %15 kısa) |
| `v1#0` ↔ `kaggle` | makine/zaman kayması + Kaggle'ın taban olarak geçerliliği |

### Tasarım notları (önemli)

1. `resolve()` sorgu başına **bir kez** koşar; tüm varyantlar **aynı aday havuzunu** görür.
   Yani varyantlar arasında yalnız prompt değişir, havuz değişmez.
2. Tekrarlar (`v1#1`) **ayrı bir geçişte** koşar. Peş peşe tekrar Ollama'nın KV-cache'ini
   yeniden kullanıyor (yerelde ölçüldü: 21,97 sn → 2,03 sn) ve çıktı yapay olarak birebir
   aynı çıkıyor — gürültüyü sıfır gösterirdi.
3. **Tek işçi** (paralellik yok). `OLLAMA_NUM_PARALLEL>1` sürekli batching yüzünden
   kendisi gürültü ekleyebilir; tam da ölçmeye çalıştığımız şey bu, karıştırmıyoruz.

### Ön koşul (Drive'da hazır olmalı)
- `institution_resolver_v3/data_processed/` → `parent_canonical.jsonl`, `subunit_canonical.jsonl`, `embeddings.npz`
- `institution_resolver_v3/ollama_models/` → `gemma4:e4b` (public registry'de YOK)

- `institution_resolver_v3/jobs/` → `faz0_ornek_125.csv` (**yerelde üretildi, Drive'a yüklenmeli**)

Ölçüm seti repoya **konmuyor** (repo herkese açık, gerçek sorgular içeriyor).
Drive'da diğer girdi dosyalarıyla aynı yerde (`jobs/`) durur; notebook onu
çalışma ağacındaki `data/eval/` yoluna kopyalar — böylece yerelde ve Colab'de
aynı repo-içi yol geçerli olur.




## 1) GPU


In [ ]:
!nvidia-smi


## 2) Drive + yollar


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT      = '/content/drive/MyDrive/institution_resolver_v3'
DRIVE_PROCESSED = f'{DRIVE_ROOT}/data_processed'
DRIVE_JOBS      = f'{DRIVE_ROOT}/jobs'        # girdi dosyalari burada (diger notebook'larla ayni)
DRIVE_OLLAMA    = f'{DRIVE_ROOT}/ollama_models'
DRIVE_OUTPUT    = f'{DRIVE_ROOT}/output'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for pth, ad in ((DRIVE_PROCESSED,'data_processed'), (DRIVE_OLLAMA,'ollama_models')):
    if not os.path.isdir(pth) or not os.listdir(pth):
        raise RuntimeError(f'Drive eksik: {ad} -> {pth}')

ORNEK_DRIVE = f'{DRIVE_JOBS}/faz0_ornek_125.csv'
if not os.path.exists(ORNEK_DRIVE):
    raise RuntimeError(
        f'Olcum seti Drive`da yok: {ORNEK_DRIVE}\n'
        'Yerelde uretilen data/eval/faz0_ornek_125.csv dosyasini Drive`daki jobs/ klasorune yukleyin.')
print('Drive hazir. Olcum seti:', ORNEK_DRIVE)


## 3) Kod


In [ ]:
REPO_DIR = '/content/institution_resolver_v3'
BRANCH   = 'feat/gate-asama1'

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}


## 4) Drive sembolik linkleri


In [ ]:
import os, shutil

def _link(name, target):
    os.makedirs(target, exist_ok=True)
    link = f'data/{name}'
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for fn in os.listdir(link):
            src, dst = f'{link}/{fn}', f'{target}/{fn}'
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
_link('processed', DRIVE_PROCESSED)
_link('jobs', DRIVE_JOBS)

# Olcum seti Drive'da jobs/ altinda durur (girdi dosyasi konvansiyonu), ama repo
# ICINDE data/eval/ yolunda olmali - judge_ab.py'nin varsayilan yolu o. Kopyalanir
# (symlink DEGIL): yerel ve Colab'de AYNI repo-ici yol gecerli olsun, hicbir komut
# --sample bayragi istemesin, `run` ile `diff` farkli dosya okuyamasin.
os.makedirs('data/eval', exist_ok=True)
shutil.copy(ORNEK_DRIVE, 'data/eval/faz0_ornek_125.csv')

!ls -la data/processed | head -5
!wc -l data/eval/faz0_ornek_125.csv


## 5) Elasticsearch

Hakem, `resolve()`'un ürettiği aday havuzuna ihtiyaç duyuyor — judge-only bir koşuda bile ES + indeks şart.


In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3


In [ ]:
import time, requests
for _ in range(60):
    try:
        r = requests.get('http://localhost:9200/_cluster/health', timeout=2)
        if r.status_code == 200:
            print('ES saglikli:', r.json()['status']); break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('ES baslatilamadi - /content/es/es.log')


## 6) Ollama + gemma4:e4b

`OLLAMA_NUM_PARALLEL=1` **bilinçli**: paralellik sürekli batching yüzünden kendisi
kararsızlık ekleyebilir, ölçtüğümüz şey tam da bu.


In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd
if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi
ollama --version


In [ ]:
import os, subprocess, time, requests, shutil

MODEL_TAG = 'gemma4:e4b'
LOCAL_OLLAMA = '/content/ollama_local'

if not os.path.isdir(LOCAL_OLLAMA):
    print('Model Drive -> yerel SSD kopyalaniyor (1-2 dk)...')
    shutil.copytree(DRIVE_OLLAMA, LOCAL_OLLAMA)

os.environ['OLLAMA_MODELS'] = LOCAL_OLLAMA
os.environ['OLLAMA_NUM_PARALLEL'] = '1'   # olcumun temizligi icin - bkz. baslik notu
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'

subprocess.run(['pkill','-f','ollama'], capture_output=True); time.sleep(2)
subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','a'),
                 stderr=subprocess.STDOUT, env=os.environ)

for _ in range(30):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).status_code == 200:
            print('Ollama ayakta (NUM_PARALLEL=1, KEEP_ALIVE=-1)'); break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama baslatilamadi')

r = requests.post('http://localhost:11434/api/generate',
                  json={'model': MODEL_TAG, 'prompt': 'Merhaba', 'stream': False})
print('warm-up:', 'OK' if r.status_code == 200 else r.text[:200])


## 7) Paketler


In [ ]:
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e '.[dev,embed,llm,api]'


## 8) İndeksleme


In [ ]:
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings


## 9) Duman testi — tezgah çalışıyor mu?

3 sorgu × 3 çağrı. Buradaki asıl kontrol **kararlar değil**, tezgahın garantileri:
havuz tek mi, `v1` tekrarlarının prompt'u aynı mı, `v1`≠`v3` mü.


In [ ]:
!python3 scripts/judge_ab.py run --variants v1,v1,v3 --limit 3 --out output/ab_smoke.csv


In [ ]:
import csv
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open('output/ab_smoke.csv', encoding='utf-8')))
for q in dict.fromkeys(r['query'] for r in rows):
    g   = [r for r in rows if r['query'] == q]
    pools = {r['candidates_json'] for r in g}
    v1h   = {r['prompt_sha256'] for r in g if r['variant'] == 'v1'}
    v3h   = {r['prompt_sha256'] for r in g if r['variant'] == 'v3'}
    print(f"{q[:36]:<38} havuz_tek={len(pools)==1}  v1_prompt_tek={len(v1h)==1}  v1!=v3={v1h!=v3h}")
print()
for r in rows:
    print(f"{r['query'][:32]:<34} {r['variant']}#{r['repeat_idx']}  {r['judge_s']:>6}s  {r['parent_verdict']}")


## 10) TAM KOŞU — 125 sorgu × 3 çağrı

İki geçiş: geçiş 0'da her sorgu için `v1`+`v3`, geçiş 1'de `v1` tekrarı.
Kaggle hızıyla (~7 sn/çağrı) kabaca **45 dakika**.

Kesinti olursa: aşağıdaki yedekleme hücresini çalıştırıp bu hücreyi `--resume` ile tekrar koş.


In [ ]:
LOCAL_OUT  = 'output/ab_faz0.csv'
DRIVE_OUT  = f'{DRIVE_OUTPUT}/ab_faz0.csv'

import os, shutil
if os.path.exists(DRIVE_OUT) and not os.path.exists(LOCAL_OUT):
    shutil.copy(DRIVE_OUT, LOCAL_OUT)   # resume icin onceki ilerleme

RESUME = '--resume' if os.path.exists(LOCAL_OUT) else ''
!python3 scripts/judge_ab.py run --variants v1,v1,v3 --out {LOCAL_OUT} {RESUME}


In [ ]:
import shutil
shutil.copy(LOCAL_OUT, DRIVE_OUT)
print('Drive yedegi:', DRIVE_OUT)


## 11) Ölçümler


### 11a) Gürültü tabanı — `v1#0` ↔ `v1#1`

Aynı prompt, aynı havuz, aynı oturum. **Bundan sonraki her A/B bu sayıyla okunacak.**

- `< %2` → tekli koşu yeterli
- `%2–10` → çift koşu zorunlu, yalnız tutarlı değişiklikler sayılır
- `> %10` → prompt A/B'si bu ölçekte terk edilir


In [ ]:
!python3 scripts/judge_ab.py diff --run {LOCAL_OUT} --a v1#0 --b v1#1 --out output/fark_gurultu.csv


### 11b) v3 etkisi — `v1#0` ↔ `v3#0`

Yanlışlanabilir tahmin: v3 **daha düşük `auto_match`** oranı göstermeli.
Fark, 11a'daki gürültü tabanından belirgin biçimde büyük değilse **bulgu yoktur**.


In [ ]:
!python3 scripts/judge_ab.py diff --run {LOCAL_OUT} --a v1#0 --b v3#0 --out output/fark_v1_v3.csv


### 11c) Kaggle tabanı — `v1#0` ↔ `kaggle`

Makine + zaman kayması. `auto_match`/`no_match` sınıflarında `<%5` ise Kaggle çıktısı
sonraki A/B'lerde taban olarak kullanılabilir → her deney tek koşuya iner.


In [ ]:
!python3 scripts/judge_ab.py diff --run {LOCAL_OUT} --a v1#0 --b kaggle --out output/fark_kaggle.csv


### 11d) Karar dağılımı ve süre


In [ ]:
import csv, collections, statistics as st
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open(LOCAL_OUT, encoding='utf-8')))

print(f"{'varyant':<8} {'auto':>5} {'review':>7} {'ambig':>6} {'nomatch':>8} {'HATA':>5} {'prompt':>8} {'medyan sn':>10}")
for key in sorted({(r['variant'], r['repeat_idx']) for r in rows}):
    g = [r for r in rows if (r['variant'], r['repeat_idx']) == key]
    c = collections.Counter(r['parent_verdict'] if r['status']=='ok' else 'HATA' for r in g)
    ch = st.median([int(r['prompt_chars']) for r in g if r['prompt_chars']])
    sn = st.median([float(r['judge_s']) for r in g if r['judge_s']])
    print(f"{key[0]}#{key[1]:<5} {c['auto_match']:>5} {c['review']:>7} {c['ambiguous']:>6} "
          f"{c['no_match']:>8} {c['HATA']:>5} {ch:>8.0f} {sn:>10.2f}")

print()
print('HATA turleri:')
for k, v in collections.Counter(r['error'][:60] for r in rows if r['status']!='ok').most_common():
    print(f'  {v:>4}  {k}')


### 11e) Fark dosyalarını Drive'a kopyala


In [ ]:
import shutil, os
for fn in ('fark_gurultu.csv', 'fark_v1_v3.csv', 'fark_kaggle.csv'):
    p = f'output/{fn}'
    if os.path.exists(p):
        shutil.copy(p, f'{DRIVE_OUTPUT}/{fn}')
        print('->', f'{DRIVE_OUTPUT}/{fn}')


---
## Sonucu nasıl okumalı

**`judge_s` varyantlar arası doğrudan karşılaştırılamaz.** Prompt'lar ortak önek
paylaştığı için aynı sorguda ikinci varyant kısmi önbellekten yararlanır. v3'ün
hız kazancı ayrı bir ölçüm ister; **kararlar bundan etkilenmez.**

**Hata sayısı tek başına yanıltıcı bir metriktir.** Bir varyantta hata azalıp
yanlışlar sessizce `auto_match` içine dağılabilir — fark dosyalarındaki
değişen kararlar tek tek incelenmeden sonuç çıkarılmamalı.

**Gürültü tabanından küçük fark = bulgu yok.** 11b/11c'yi 11a olmadan okuma.
